# Exercice

Creez un preprocessing pour des avis de films avec emojis, mentions d'utilisateurs (@user), et URLs. Quel est le meilleur ordre d'etapes ?

In [51]:
sentences = [
    "Ce film était incroyable 😍 ! Regardez la bande-annonce ici : https://www.youtube.com/watch?v=abc123",
    "@critique_cine Je suis totalement d'accord, ce film était une vraie catastrophe 😤😡",
    "@user123 Tu as vu ce film ? 🎬🍿 Plus d'infos sur http://www.allocine.fr/film/fichefilm_gen_cfilm=12345.html",
    "La fin du film... 😭😭😭 je n'ai pas pu dormir de la nuit",
    "Meilleur film de l'année selon www.imdb.com/title/tt1234567, un chef-d'œuvre absolu 🏆✨",
    "@dupont @martin Vous recommandez ce film ? https://t.co/xYz789 les critiques sont dingues 🔥",
    "Un scénario décevant, les personnages manquent de profondeur. Pas convaincu.",
    "ABSOLUMENT MAGNIFIQUE !!! 😲🤩 Le meilleur film que j'ai vu depuis des années !!!",
    "@filmfan99 INCROYABLE ce film 😍🎥 ! Voir la critique complète : https://www.senscritique.com/film/review/123",
    "Nul à mourir... 🥱😴💤 deux heures de ma vie perdues, je veux un remboursement !!!",
]

# Solution

In [52]:
from typing import Callable
import re
import string
import nltk
from nltk.corpus import stopwords


def ensure_stopwords_downloaded():
    try:
        nltk.data.find("corpora/stopwords")
    except LookupError:
        nltk.download("stopwords")


ensure_stopwords_downloaded()

In [53]:
def to_lower(sentence: str) -> str:
    return sentence.lower()

In [54]:
URL_PATTERN = (
    r"\b(?:https?://)?"
    r"(?:www\.)?"
    r"(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}"
    r"(?::\d+)?"
    r"(?:/[^\s?#]*)?"
    r"(?:\?[^\s#]*)?"
    r"(?:#[^\s]*)?"
)


def remove_urls(sentence: str) -> str:
    return re.sub(URL_PATTERN, "", sentence)

In [55]:
USER_PATTERN = r"@\S+"


def remove_users(sentence: str) -> str:
    return re.sub(USER_PATTERN, "", sentence)

In [56]:
PUNCTUATIONS = string.punctuation


def remove_punctuations(sentence: str) -> str:
    return re.sub(f"[{re.escape(PUNCTUATIONS)}]", " ", sentence)

In [57]:
STOP_WORDS = set(stopwords.words("french"))


def remove_stop_words(sentence: str) -> str:
    words = sentence.split()
    filtered_words = [word for word in words if word.lower() not in STOP_WORDS]
    return " ".join(filtered_words)

In [58]:
EMOJI_PATTERN = re.compile(
    "["
    "\U0001f600-\U0001f64f"
    "\U0001f300-\U0001f5ff"
    "\U0001f680-\U0001f6ff"
    "\U0001f1e0-\U0001f1ff"
    "\U00002700-\U000027bf"
    "\U0001f900-\U0001f9ff"
    "\U00002600-\U000026ff"
    "\U00002b00-\U00002bff"
    "]+",
    flags=re.UNICODE,
)


def remove_emojis(sentence: str) -> str:
    return EMOJI_PATTERN.sub("", sentence)

In [59]:
def preprocessing(sentences: list[str], transformers: list[Callable]) -> list[str]:
    sentences_cleaned = []
    for sentence in sentences:
        for transformer in transformers:
            sentence = transformer(sentence)
        sentences_cleaned.append(sentence)
    return sentences_cleaned


transformers = [
    to_lower,
    remove_urls,
    remove_users,
    remove_punctuations,
    remove_emojis,
    remove_stop_words,
]

preprocessing(sentences, transformers)

['film incroyable regardez bande annonce ici',
 'totalement accord film vraie catastrophe',
 'vu film plus infos',
 'fin film pu dormir nuit',
 'meilleur film année selon chef œuvre absolu',
 'recommandez film critiques dingues',
 'scénario décevant personnages manquent profondeur convaincu',
 'absolument magnifique meilleur film vu depuis années',
 'incroyable film voir critique complète',
 'nul mourir deux heures vie perdues veux remboursement']